In [21]:
"""
0. Loads three CSV files (`event_bd.csv`, `occurrence_bd.csv`, and `emof_bd.csv`) into pandas
DataFrames.
1. Checks for the presence of required columns;
2. Verifies data completeness by looking for null values in critical fields;
3. Validates the geographic coordinates (latitude and longitude) to ensure they are within valid
   ranges;
4. Checks for depth information, making sure that `minimumDepthInMeters` is not greater than
   `maximumDepthInMeters` and that depth values are numeric;
5. Checks with the World Register of Marine Species (WoRMS) API to validate scientific names,
   identifying any unaccepted or unfound taxa.
6. Then attempts to merge these DataFrames.
   a. First, `df_event` is merged with `df_occurrence` on the `eventID` column, with a validation
      to ensure it's a one-to-many relationship. The event file should have a unique list of
      `eventID` which match to `eventID` in the occurrence file.
   b. Next, the resulting DataFrame (`df_event_occur`) is merged with df_emof on occurrenceID,
      again with a one-to-many validation. The event and occurrence data files should have unique
      `occurrenceID` values which map to the `occurrenceID` in the emof file.
7. Finally, it checks if the number of rows in the final merged DataFrame (df_event_occur_emof)
   is equal to the number of rows in the df_emof DataFrame, indicating a successful arrangement
   of event files. Since we are merging all of the data together, there should be the same number
   of rows in the final dataset as there are in the emof file, but with more columns from the
   event and occurrence files.

"""

##-----------------------------------##
##             Libraries             ##
##-----------------------------------##
import functools

import stamina

import pandas as pd
import janitor  # noqa: F401
import requests
from pathlib import Path

import numpy as np

##-----------------------------------##
##          Create Functions         ##
##-----------------------------------##

# -------- read --------


# -------- required_event_columns --------

def required_event_columns(df):
     """
    check event core columns against the required DwC terms

    @param df        A pandas dataframe of event core
 
    @return          A list of missing required DwC columns
    """

    # set the required event core columns
    required_event_columns = [
    "eventID",
    "eventDate",
    "decimalLatitude",
    "decimalLongitude",
    "countryCode",
    "geodeticDatum",
]
    # compare to the event df and print what required columns are missing
    missing_cols = list(set(required_event_columns).difference(df.columns))
    res = True
    msg = "✅ Passed required columns!"
    if missing_cols:
        msg = f"❌ Failed! Missing required DwC {missing_cols} columns."
        res = False
    return res, msg

# -------- required_occ_columns --------

def required_occ_columns(df):
    """
    check occurrence extension columns against the required DwC terms

    @param df        A pandas dataframe of occurrence extension
 
    @return          A list of missing required DwC columns
    """

    # set required occurrence columns
    required_occ_columns = [
    "occurrenceID",
    "scientificName",
    "eventDate",
    "decimalLatitude",
    "decimalLongitude",
    "basisOfRecord",
    "occurrenceStatus",
]
    # compare to occurrence df and print a list of missing required occurrence columns
    missing_cols = list(set(required_occ_columns).difference(df.columns))
    res = True
    msg = "✅ Passed required columns!"
    if missing_cols:
        msg = f"❌ Failed! Missing required DwC {missing_cols} columns."
        res = False
    return res, msg

# -------- required_event_columns --------

def required_emof_columns(df):
    """
    check emof columns against the required DwC terms

    @param df        A pandas dataframe of emof
 
    @return          A list of missing required DwC columns
    """

    # set required emof columns
    required_emof_columns = [
    "eventID",
    "occurrenceID",
    "measurementValue",
    "measurementType",
    "measurementUnit",
]
    # compare to the emof df and print missing required emof columns
    missing_cols = list(set(required_emof_columns).difference(df.columns))
    res = True
    msg = "✅ Passed required columns!"
    if missing_cols:
        msg = f"❌ Failed! Missing required DwC {missing_cols} columns."
        res = False
    return res, msg

# -------- check_dwc_columns --------
# load in the list of terms for event, ocurrence, and emof
TDWG_BASE = "https://raw.githubusercontent.com/tdwg/dwc/master/dist/dwc-a/"
TERM_LISTS = {
    "event": f"{TDWG_BASE}dwc_event_core_vertical.csv",
    "occurrence": f"{TDWG_BASE}dwc_occurrence_core_vertical.csv",
    "emof": f"{TDWG_BASE}measurements_or_facts_extension_vertical.csv",
}

# -------- check_occ_dupes --------
def check_occ_dupes(df):
    """
    Check if occurrenceID has any duplicate values in the occurrence dataframe.

    @param df        A pandas dataframe of the occurrence extension

    @return          A list of non-unique occurrenceIDs
    """
    ids = df["occurrenceID"]
    occ_dupes = ids[ids.duplicated()].unique().tolist()

    if occ_dupes:
        print(f"❌ Duplicate occurrenceIDs found: {occ_dupes}")
    else:
        print("✅ No duplicate occurrenceIDs!")

# -------- check_event_dupes --------
def check_event_dupes(df):
    """
    Check if eventID has any duplicate values in the event dataframe.

    @param df        A pandas dataframe of the event core

    @return          A list of non-unique eventIDs
    """
    ids = df["eventID"]
    event_dupes = ids[ids.duplicated()].unique().tolist()

    if event_dupes:
        print(f"❌ Duplicate eventIDs found: {event_dupes}")
    else:
        print("✅ No duplicate eventIDs!")

# -------- check_merge_tables --------

def check_merge_tables(df_event, df_occurrence, df_emof):
    """
    Merge event -> occurrence -> emof, validating one-to-many at each step.

    @param df_event        A pandas dataframe of event core
    @param df_occurrence   A pandas dataframe of occurrence extension
    @param df_emof         A pandas dataframe of emof

    @return merged table or errors from merge
    """

    try:
        df_event_occur = df_event.merge(
            df_occurrence, on="eventID", validate="one_to_many", suffixes=(None, "_occ")
        )
        if df_event_occur.empty:
            return None, "❌ Merge event/occurrence tables failed: no matching eventIDs."

        # Treat empty or whitespace-only occurrenceIDs as missing.
        if "occurrenceID" in df_emof.columns:
            df_emof = df_emof.assign(
                occurrenceID=df_emof["occurrenceID"].replace(r"^\s*$", np.nan, regex=True)
            )

        # Use occurrenceID if emof has any values in it; otherwise fall back to eventID.
        if "occurrenceID" in df_emof.columns and df_emof["occurrenceID"].notna().any():
            key = "occurrenceID"
        else:
            key = "eventID"
            df_emof = df_emof.drop(columns="occurrenceID", errors="ignore")
            df_event_occur = df_event_occur.drop_duplicates("eventID")

        df = df_event_occur.merge(
            df_emof, on=key, validate="one_to_many", suffixes=(None, "_emof")
        )
        if df.empty:
            return None, f"❌ Merge emof table failed: no matching {key} values."

        msg = f"✅ Merge tables passed (emof joined on {key})!"
        if len(df) != len(df_emof):
            msg += f"\n⚠️  Merged rows ({len(df)}) != emof rows ({len(df_emof)})."
        return df, msg
    except pd.errors.MergeError as err:
        return None, f"❌ Merge tables failed.\n{err}"


# -------- check_null_values --------

def check_null_values(df):
    """
    check the merged dataframe for null values accross all columns

    @param df        A pandas dataframe of all merged tables (output of check_merge_tables)    
 
    @return          A list of columns that contain null values
    """
    
    required_columns = [
    "eventID",
    "occurrenceID",
    "eventDate",
    "decimalLatitude",
    "decimalLongitude",
    "countryCode",
    "geodeticDatum",
    "scientificName",
    "basisOfRecord",
    "occurrenceStatus",
    "measurementValue",
    "measurementType",
    "measurementUnit",
]
    missing = df.columns[df.isna().any()].to_list()
    res = True
    msg = "✅ Passed null values check!"
    if missing:
        msg = f"⚠️  Columns {missing} have missing values."
        res = False
    return res, msg

# -------- check_latitude --------

def check_latitude(df):
    """
    check to make sure that latitude is within bounds

    @param df        A pandas dataframe that contains decimalLatitude    
 
    @return          A list of invalid decimalLatitude values
    """
    
    res = True
    msg = "✅ Passed `decimalLatitude` bounds!"
    if "decimalLatitude" not in df.columns:
        return False, "⚠️  Cannot find `decimalLatitude` column."
    invalid_lat = df[
        pd.to_numeric(df["decimalLatitude"], errors="coerce").isna()
        | (df["decimalLatitude"] <= -90)
        | (df["decimalLatitude"] >= 90)
    ]
    if not invalid_lat.empty:
        msg = f"❌ Invalid `decimalLatitude` values detected. {invalid_lat.index.tolist()}"
        res = False
    return res, msg

# -------- check_longitude --------

 """
    check to make sure that longitude is within bounds

    @param df        A pandas dataframe that contains decimalLongitude
 
    @return          A list of invalid decimalLongitude values
    """

def check_longitude(df):
    res = True
    msg = "✅ Passed `decimalLongitude` bounds!"
    if "decimalLongitude" not in df.columns:
        return False, "⚠️  Cannot find `decimalLongitude` column."
    invalid_lon = df[
        pd.to_numeric(df["decimalLongitude"], errors="coerce").isna()
        | (df["decimalLongitude"] <= -180)
        | (df["decimalLongitude"] >= 180)
    ]
    if not invalid_lon.empty:
        msg = f" Invalid decimalLongitude values detected. {invalid_lon.index.tolist()}"
        res = False
    return res, msg

# -------- check_depth_consistency --------

def check_depth_consistency(df):
    """
    check to make sure depth values are consistent

    @param df        A pandas dataframe that contains minimum and maximum depth in meters
 
    @return          A list of columns that contain incorrect depth values
    """
    
    res = True
    msg = "✅ Passed depth consistency test!"
    if ("minimumDepthInMeters", "maximumDepthInMeters") not in df.columns:
        msg = (
            "⚠️  No depth information found (minimumDepthInMeters/maximumDepthInMeters)."
        )
        # We return here b/c we cannot run the tests below without these columns.
        return False, msg

    min_depth = pd.to_numeric(df["minimumDepthInMeters"], errors="coerce")
    msg_min = ""
    if not min_depth.isna().empty:
        msg_min = (
            f"⚠️  Non-numeric values in minimumDepthInMeters {min_depth.index.tolist()}"
        )
        res = False

    max_depth = pd.to_numeric(df["minimumDepthInMeters"], errors="coerce")
    msg_max = ""
    if not max_depth.isna().empty:
        msg_max = (
            f"⚠️  Non-numeric values in minimumDepthInMeters {max_depth.index.tolist()}"
        )
        res = False

    # Check logic: Min should not be greater than Max
    illogical = all(min_depth >= max_depth)

    if not illogical.empty:
        msg = f"❌ minimumDepthInMeters is greater than maximumDepthInMeters {illogical.tolist()}"
        res = False
    msg = f"{msg_min}\n{msg_max}\n{msg}"
    return res, msg

# -------- check_scientific_names --------

def check_scientific_names(df):
    """
    check scientific names against WoRMS

    @param df        A pandas dataframe that contains species information   
 
    @return          A list of inconsistent scientific naming against WoRMS
    """
    
    if "scientificName" not in df.columns:
        return [None, "⚠️  Missing the `scientificName` column!"]

    names = list(set(df["scientificName"]))

    results = [check_scientific_name(name) for name in names]
    return [msg for res, msg in results if not res]


@functools.lru_cache(maxsize=128)
def check_scientific_name(name):
    response = _check_scientific_name(name)

    # Bail early to avoid unnecessary retries.
    if response.status_code == 204 or response.status_code == 400:
        return False, f"⚠️  {response.status_code=} for {name=}."

    if response.status_code == 200:
        results = response.json()
    else:
        msg = f"⚠️  WoRMS API Error: {response.status_code} for {name=}"
        return False, msg

    # The API returns a list of lists (one list per name queried).
    is_unique = f"Found 1 match for {name=}"
    if len(results) > 1:
        is_unique = "⚠️  Found more than 1 match for {name=}, selecting the first one"

    # Take first match.
    result = results[0]
    if result["status"] != "accepted":
        msg = f"{is_unique}\n⚠️  Taxon {result['status']=}. Accepted name: {result['valid_name']=}, {result['url']=}"
        return False, msg
    return True, f"{is_unique}."


@stamina.retry(on=requests.exceptions.HTTPError, attempts=3)
def _check_scientific_name(name):
    url = f"http://www.marinespecies.org/rest/AphiaRecordsByName/{name}?like=true&marine_only=true"
    return requests.get(url, timeout=60)

In [22]:
base_url = 'https://github.com/ioos/bio_data_guide/raw/refs/heads/main/datasets/example_script_with_fake_data/'
good_data = {
        'occur': f'{base_url}MadeUpData_Occurrence.csv',
        'event': f'{base_url}MadeUpData_event.csv',
        'emof': f'{base_url}MadeUpData_mof.csv'
    }
df_event = pd.read_csv(good_data['event'], **kw).clean_names(
        **clean_names
    )
df_occurrence = pd.read_csv(
        good_data['occur'], **kw
    ).clean_names(**clean_names)
df_emof = pd.read_csv(good_data['emof'], **kw).clean_names(
        **clean_names
    )

check_emof_columns(df_emof)

check_merge_tables(
        df_event=df_event, df_occurrence=df_occurrence, df_emof=df_emof
    )

(None, '❌ Merge emof table failed: no matching occurrenceID values.')

In [24]:
if __name__ == "__main__":


    base_url = 'https://github.com/ioos/bio_data_guide/raw/refs/heads/main/datasets/example_script_with_fake_data/'
    good_data = {
        'occur': f'{base_url}MadeUpData_Occurrence.csv',
        'event': f'{base_url}MadeUpData_event.csv',
        'emof': f'{base_url}MadeUpData_mof.csv'
    }
    
    # TODO: Web version add 3 boxes for each upload.
    # test_dir = Path("tests/data/bad_data")
    test_dir = Path("tests/data/good_data")
    # test_dir = Path("tests/data/encoding_issues")

    kw = {"encoding_errors": "ignore"}
    clean_names = {
        "axis": "columns",
        "strip_underscores": True,
        "case_type": "preserve",
        "remove_special": True,
    }

    df_event = pd.read_csv(good_data['event'], **kw).clean_names(
        **clean_names
    )
    df_occurrence = pd.read_csv(
        good_data['occur'], **kw
    ).clean_names(**clean_names)
    df_emof = pd.read_csv(good_data['emof'], **kw).clean_names(
        **clean_names
    )

    df, msg = check_merge_tables(
        df_event=df_event, df_occurrence=df_occurrence, df_emof=df_emof
    )
    print(msg)

    if df is not None:
        res, msg = check_required_columns(df, columns=required_columns)
        print(msg)

        res, msg = check_null_values(df, columns=required_columns)
        print(msg)

        res, msg = check_latitude(df)
        print(msg)

        res, msg = check_longitude(df)
        print(msg)

        res, msg = check_depth_consistency(df)
        print(msg)

        results = check_scientific_names(df)
        [print(msg) for msg in results]


❌ Merge emof table failed: no matching occurrenceID values.
